# Multi-Indicator Trading Strategies

This notebook demonstrates:
1. Combining multiple technical indicators
2. Using pre-built strategy library
3. Comparing different strategy approaches
4. Building complex trading logic

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import kimsfinance_core
from kimsfinance.strategies import (
    RSIStrategy, MACDStrategy, EMACrossoverStrategy,
    ATRBreakoutStrategy, BollingerBreakoutStrategy
)
from kimsfinance.visualization import plot_performance_dashboard, print_performance_summary

print(f"kimsfinance_core v{kimsfinance_core.__version__}")

## 1. Generate Test Data

In [ ]:
def generate_sample_data(n=2000, seed=42):
    np.random.seed(seed)
    timestamps = np.arange(n, dtype=np.int64) * 60
    base = np.linspace(100.0, 180.0, n)
    noise = np.random.randn(n).cumsum() * 3
    close = base + noise
    open_prices = close + np.random.randn(n) * 0.5
    high = np.maximum(open_prices, close) + np.abs(np.random.randn(n) * 2)
    low = np.minimum(open_prices, close) - np.abs(np.random.randn(n) * 2)
    volume = np.random.uniform(1000, 10000, n)
    return timestamps, open_prices, high, low, close, volume

timestamps, open_p, high, low, close, volume = generate_sample_data(2000)
print(f"Generated {len(close)} candles")

## 2. Test Multiple Strategies

Let's compare different strategy types on the same data.

In [ ]:
strategies = [
    ('RSI Mean Reversion', RSIStrategy(period=14, buy_threshold=30, sell_threshold=70)),
    ('EMA Crossover', EMACrossoverStrategy(fast_period=12, slow_period=26)),
    ('ATR Breakout', ATRBreakoutStrategy(period=14, multiplier=2.0)),
]

results = {}

for name, strategy in strategies:
    print(f"\nTesting {name}...")
    
    result = kimsfinance_core.run_backtest(
        high=high, low=low, close=close, open_prices=open_p,
        volume=volume, timestamps=timestamps, strategy=strategy,
        initial_capital=10000.0, trading_fee=0.001,
        slippage=0.0005, use_gpu=False
    )
    
    results[name] = result
    print_performance_summary(result)

## 3. Compare Strategy Performance

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    name: {
        'Total Return (%)': result['total_return'],
        'Sharpe Ratio': result['sharpe_ratio'],
        'Max Drawdown (%)': result['max_drawdown'],
        'Win Rate (%)': result['win_rate'],
        'Num Trades': result['num_trades'],
        'Profit Factor': result['profit_factor']
    }
    for name, result in results.items()
}).T

print("\nStrategy Comparison:")
print(comparison)

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sharpe ratio
comparison['Sharpe Ratio'].plot(kind='bar', ax=axes[0,0], color='#2E86AB')
axes[0,0].set_title('Sharpe Ratio Comparison', fontweight='bold')
axes[0,0].set_ylabel('Sharpe Ratio')
axes[0,0].grid(True, alpha=0.3)

# Total return
comparison['Total Return (%)'].plot(kind='bar', ax=axes[0,1], color='#F18F01')
axes[0,1].set_title('Total Return Comparison', fontweight='bold')
axes[0,1].set_ylabel('Return (%)')
axes[0,1].grid(True, alpha=0.3)

# Max drawdown
comparison['Max Drawdown (%)'].plot(kind='bar', ax=axes[1,0], color='#A23B72')
axes[1,0].set_title('Max Drawdown Comparison', fontweight='bold')
axes[1,0].set_ylabel('Drawdown (%)')
axes[1,0].grid(True, alpha=0.3)

# Win rate
comparison['Win Rate (%)'].plot(kind='bar', ax=axes[1,1], color='#18A558')
axes[1,1].set_title('Win Rate Comparison', fontweight='bold')
axes[1,1].set_ylabel('Win Rate (%)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Compare Equity Curves

In [ ]:
plt.figure(figsize=(14, 6))

for name, result in results.items():
    plt.plot(result['equity_curve'], label=name, linewidth=2)

plt.title('Strategy Equity Curve Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Bars', fontsize=12)
plt.ylabel('Equity ($)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Create Custom Multi-Indicator Strategy

Let's build a strategy that combines RSI and trend filters.

In [ ]:
class RSIWithTrendFilter:
    """
    RSI strategy with EMA trend filter
    - Only buy when price > EMA (uptrend)
    - Use RSI for entry/exit timing
    """
    def __init__(self, rsi_period=14, ema_period=50, buy_threshold=30, sell_threshold=70):
        self.rsi_period = rsi_period
        self.ema_period = ema_period
        self.buy_threshold = buy_threshold
        self.sell_threshold = sell_threshold
    
    def on_data(self, bar, indicators):
        rsi = indicators.get(f'rsi_{self.rsi_period}', 50.0)
        ema = indicators.get(f'ema_{self.ema_period}', bar['close'])
        price = bar['close']
        
        # Trend filter: only trade in uptrend
        in_uptrend = price > ema
        
        if in_uptrend and rsi < self.buy_threshold:
            return 'buy'
        elif rsi > self.sell_threshold:
            return 'sell'
        return 'hold'
    
    def get_indicators(self):
        return [f'rsi_{self.rsi_period}', f'ema_{self.ema_period}']

# Test custom strategy
custom_strategy = RSIWithTrendFilter()

custom_result = kimsfinance_core.run_backtest(
    high=high, low=low, close=close, open_prices=open_p,
    volume=volume, timestamps=timestamps, strategy=custom_strategy,
    initial_capital=10000.0, trading_fee=0.001,
    slippage=0.0005, use_gpu=False
)

print("\nCustom RSI + Trend Filter Strategy:")
print_performance_summary(custom_result)

## 6. Next Steps

Ideas for more advanced strategies:

1. **Machine Learning Integration**: Use ML models for signal generation
2. **Portfolio Strategies**: Allocate between multiple strategies
3. **Risk Management**: Add stop-loss, take-profit, position sizing
4. **Market Regime Detection**: Adapt strategy based on market conditions
5. **Multi-Timeframe Analysis**: Combine signals from different timeframes